In [1]:
%pip install cv2

ERROR: Could not find a version that satisfies the requirement cv2 (from versions: none)
ERROR: No matching distribution found for cv2


# 🧠 Training Image Classification với ResNet50 (Pytorch)

Notebook này thực hiện huấn luyện mô hình phân loại ảnh sử dụng ResNet50 (Train from Scratch).
Các điểm nổi bật:
1. **Tối ưu Data Loading**: Sử dụng OpenCV để đọc ảnh nhanh, lazy loading để tiết kiệm RAM.
2. **Train/Val/Test Split**: Chia dữ liệu thành 3 tập: **Train (70%)**, **Val (15%)**, **Test (15%)**.
3. **Data Augmentation**: Tăng cường dữ liệu chỉ cho tập Train.
4. **Optimizer**: Sử dụng AdamW.
5. **Early Stopping**: Dừng sớm nếu Validation Loss không giảm.
6. **Final Evaluation**: Đánh giá mô hình trên tập Test (chưa từng thấy trong quá trình train).

In [1]:
import kagglehub
import os
import time
import copy
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models, transforms

# Check device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"🔹 Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

# Create model folder

# Download dataset
path = kagglehub.dataset_download("jessicali9530/caltech256")
print("Path to dataset files:", path)

# Create model folder
model_dir = 'models'
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.abspath(model_dir)
print(f"Model directory created: {model_path}")


## 1. Cấu hình & Hyperparameters

In [ ]:
CONFIG = {
    'data_dir': os.path.join(path, '256_ObjectCategories'), # Updated to use downloaded path
    'model_dir': model_path,
    'image_size': 224,         # ResNet standard input
    'batch_size': 32,
    'num_epochs': 1,
    'learning_rate': 0.001,
    'momentum': 0.9,           # Momentum for SGD
    'weight_decay': 1e-4,      # L2 Regularization
    'num_workers': 2,
    'seed': 42,
    'test_split': 0.15,        # 15% for Test
    'val_split': 0.15,         # 15% for Val (approx 17.6% of remaining 85%)
    'patience': 5              # Early Stopping Patience
}

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

## 2. Dataset Setup (Custom Lazy Loading)

In [ ]:
# Hàm quét toàn bộ ảnh và labels
def get_image_paths(root_dir):
    image_paths = []
    labels = []
    classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
    class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
    
    print(f"🔍 Quét dữ liệu từ: {root_dir}")
    print(f"   Classes tìm thấy: {classes}")
    
    for cls_name in classes:
        cls_folder = os.path.join(root_dir, cls_name)
        files = [f for f in os.listdir(cls_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        for f in files:
            image_paths.append(os.path.join(cls_folder, f))
            labels.append(class_to_idx[cls_name])
            
    print(f"   ✅ Tổng số ảnh: {len(image_paths)}")
    return image_paths, labels, classes

# Dataset Class tùy chỉnh
class DogDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        path = self.image_paths[idx]
        label = self.labels[idx]
        
        # Đọc bằng CV2 (Nhanh hơn)
        img = cv2.imread(path)
        if img is None:
            # Fallback nếu lỗi ảnh (tạo ảnh đen)
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (CONFIG['image_size'], CONFIG['image_size']))
            
        img = Image.fromarray(img)
        
        if self.transform:
            img = self.transform(img)
            
        return img, label

## 3. Data Transformations & Loading (Train/Val/Test Split)

In [ ]:
# Lấy danh sách ảnh
all_paths, all_labels, class_names = get_image_paths(CONFIG['data_dir'])

# 1. Split Train+Val vs Test (85% vs 15%)
train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
    all_paths, all_labels, 
    test_size=CONFIG['test_split'], 
    stratify=all_labels, 
    random_state=CONFIG['seed']
)

# 2. Split Train vs Val (Từ 85% còn lại, lấy ra sao cho Val = 15% tổng)
# Val = 15% total => Val / (Train+Val) = 15 / 85 approx 0.1765
val_split_relative = CONFIG['val_split'] / (1 - CONFIG['test_split'])

train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths, train_val_labels, 
    test_size=val_split_relative, 
    stratify=train_val_labels, 
    random_state=CONFIG['seed']
)

print(f"🔹 Train size: {len(train_paths)} ({len(train_paths)/len(all_paths)*100:.1f}%)")
print(f"🔹 Val size:   {len(val_paths)} ({len(val_paths)/len(all_paths)*100:.1f}%)")
print(f"🔹 Test size:  {len(test_paths)} ({len(test_paths)/len(all_paths)*100:.1f}%)")

# Augmentation
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    # Ảnh đã được resize về 224 ở Dataset
    transforms.RandomResizedCrop(CONFIG['image_size'], scale=(0.8, 1.0)), # Enhanced Augmentation
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

# Tạo Datasets
train_dataset = DogDataset(train_paths, train_labels, transform=train_transform)
val_dataset = DogDataset(val_paths, val_labels, transform=val_test_transform)
test_dataset = DogDataset(test_paths, test_labels, transform=val_test_transform)

# Tạo DataLoaders
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, 
                          num_workers=CONFIG['num_workers'], pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, 
                        num_workers=CONFIG['num_workers'], pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, 
                        num_workers=CONFIG['num_workers'], pin_memory=True)

dataloaders = {'train': train_loader, 'val': val_loader, 'test': test_loader}
dataset_sizes = {'train': len(train_dataset), 'val': len(val_dataset), 'test': len(test_dataset)}

## 4. Model Setup (ResNet50 + AdamW)

In [ ]:
# Load Model from Scratch (No Pretrained)
model = models.resnet50(weights=None)

# Thay đổi lớp cuối cùng
num_classes = len(class_names)
in_features = model.fc.in_features

# New layers have requires_grad=True by default
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(in_features, num_classes)
)

model = model.to(device)

# Loss & Optimizer
criterion = nn.CrossEntropyLoss()
# Optimizer: AdamW, optimize all parameters
# Optimizer: SGD with Momentum
optimizer = optim.SGD(model.parameters(), lr=CONFIG['learning_rate'], 
                      momentum=CONFIG['momentum'], weight_decay=CONFIG['weight_decay'])

# Scheduler
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=3, verbose=True)

## 5. Training Loop with Early Stopping

In [ ]:
# Early Stopping Class
class EarlyStopping:
    def __init__(self, patience=5, verbose=False, delta=0):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        
        # Tạo thư mục nếu chưa có
        os.makedirs(CONFIG['model_dir'], exist_ok=True)
        
        torch.save(model.state_dict(), os.path.join(CONFIG['model_dir'], 'checkpoint.pth'))
        self.val_loss_min = val_loss

def train_model(model, criterion, optimizer, scheduler, num_epochs=25, patience=5):
    since = time.time()
    
    # Initialize Early Stopping
    early_stopping = EarlyStopping(patience=patience, verbose=True)
    
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            # Progress Bar
            pbar = tqdm(dataloaders[phase], desc=f"{phase.upper()}", leave=False)
            
            for inputs, labels in pbar:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                
                pbar.set_postfix({'loss': f"{loss.item():.4f}"})

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())
                scheduler.step(epoch_acc)
                
                # Check Early Stopping
                early_stopping(epoch_loss, model)
                
                if early_stopping.early_stop:
                    print("Early stopping")
                    # Load best model from checkpoint
                    model.load_state_dict(torch.load(os.path.join(CONFIG['model_dir'], 'checkpoint.pth')))
                    return model, history

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
        
        if early_stopping.early_stop:
             break

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')

    model.load_state_dict(best_model_wts)
    return model, history

In [ ]:
# Chạy Training
model, history = train_model(model, criterion, optimizer, scheduler, num_epochs=CONFIG['num_epochs'], patience=CONFIG['patience'])

## 6. Evaluation on Test Set

In [ ]:
print(f"\n💥 FINAL EVALUATION ON TEST SET ({len(test_dataset)} images)")

# Load Model tốt nhất từ Training hoặc Early Stopping
# Nếu train hết mà không early stop, model đã là best model
# Nếu early stop, model đã load checkpoint
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in tqdm(dataloaders['test'], desc="TESTING"):
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("\nClassification Report (Test Set):")
print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

# Confusion Matrix (Optional - might be too big for 256 classes)
if len(class_names) <= 50:
    plt.figure(figsize=(10, 8))
    cm = confusion_matrix(all_labels, all_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix (Test Set)')
    plt.show()
else:
    print(f"⚠️ Too many classes ({len(class_names)}) to display readable confusion matrix.")

In [ ]:
# Lưu Model
os.makedirs(CONFIG['model_dir'], exist_ok=True)
torch.save(model.state_dict(), os.path.join(CONFIG['model_dir'], 'resnet50_256_object_categories.pth'))
print("Model saved to ./models/resnet50_256_object_categories.pth")